# DPR (2020)
---
[[paper]](https://arxiv.org/abs/2004.04906)<br>
DPR = Dense Passage Retrieval (with Contrastive Training Logic)

__DPR__ — это метод <u>плотного</u> информационного поиска (Dense Retrieval), основанный на архитектуре Bi-Encoder. Решение позволяет сопоставлять запросы и документы в семантическом векторном пространстве, эффективно решая проблему "лексического разрыва" (lexical gap), когда запрос и релевантный документ не имеют общих слов.

__Постановка задачи__<br>
В контексте Open-Domain Question Answering (ODQA) необходимо из огромной коллекции (миллионы документов) быстро извлечь подмножество пассажей, которые с высокой вероятностью содержат ответ на произвольный текстовый вопрос.

__Мотивация__<br>
Традиционные системы (BM25, TF-IDF) работают на поиске точных совпадений подстрок. Если пользователь спрашивает "Who is the author of the theory of relativity?", а в тексте написано "Einstein developed the principles of space-time physics", BM25 может пропустить этот документ. Требовалось решение, которое понимает семантическое сходство, но при этом работает так же быстро, как инвертированный индекс.

__Существующие подходы__<br>
На 2020 год основными конкурентами были:
- BM25 (1990-е): высокая скорость, но нулевая семантика. Работает только на пересечении токенов.
- Cross-Encoders (например, BERT re-rankers, 2019): подают запрос и документ в модель одновременно. Очень точные (учитывают interaction между словами), но экстремально медленные — невозможно прогнать запрос через миллионы документов в реальном времени.
- ORQA (2019): первая попытка Dense Retrieval для ODQA. Использовала Inverse Cloze Task для предобучения, но архитектура была сложной в обучении и требовала огромных вычислительных ресурсов для совместного обучения ретривера и ридера.

__Идея__<br>
Использовать Bi-Encoder архитектуру, где запросы и документы кодируются <u>независимо</u> в векторы фиксированной размерности. Ключевая новизна заключается в стратегии обучения: использовании Contrastive Learning с In-batch negatives и специально подобранными Hard negatives (через BM25), что позволило модели различать семантически близкие, но нерелевантные пассажи.

__Архитектура__<br>
DPR состоит из двух независимых трансформеров:
1.  Question Encoder ($E_Q$): преобразует текст вопроса в вектор $d$-размерности. Обычно это BERT-base, где берется эмбеддинг [CLS] токена.
2.  Passage Encoder ($E_P$): преобразует текст пассажа в вектор той же размерности.
3.  Similarity Function: сходство вычисляется как простое скалярное произведение (dot product) векторов: $sim(q, p) = E_Q(q)^T E_P(p)$.

__Алгоритм обучения__<br>
Используется Contrastive Loss (Negative Log Likelihood). Для каждого вопроса в батче есть один позитивный пассаж ($p^+$) и набор негативных ($p^-$).
1.  In-batch negatives: для вопроса $q_i$ позитивным является $p_i^+$, а негативами служат все остальные позитивные пассажи других вопросов в этом же батче ($p_j^+$ при $j \neq i$). Это позволяет эффективно использовать память GPU, создавая $B-1$ негативов для каждого примера "бесплатно".
2.  Hard negatives: в батч специально подмешиваются пассажи, которые имеют высокий скор по BM25, но не содержат текст ответа. Это заставляет модель не просто искать "похожие слова", а вникать в суть вопроса.
3.  Оптимизация: минимизируется расстояние до $p^+$ и максимизируется до всех $p^-$.

__Алгоритм инференса__<br>
1.  Offline: вся коллекция документов (например, 21 млн пассажей Википедии) прогоняется через $E_P$. Полученные векторы индексируются в FAISS (библиотека для быстрого поиска ближайших соседей).
2.  Online: пришедший вопрос кодируется через $E_Q$.
3.  Retrieval: выполняется Maximum Inner Product Search (MIPS) в FAISS-индексе. Возвращается топ-$K$ документов.
Благодаря независимой кодировке, время поиска по 20 млн документов составляет единицы миллисекунд.

__Результаты__<br>
На датасете Natural Questions (NQ):
- Метрика Top-20 Retrieval Accuracy (есть ли ответ в 20 выданных пассажах): DPR показал 79.4% против 59.1% у BM25. Прирост составил более 20пп.
- В связке с моделью-ридером (End-to-end QA) точность ответов выросла на 9-11пп по сравнению с системами на базе BM25.
- DPR доказал, что плотные векторы могут полностью заменить разреженные представления в задачах поиска без потери качества на больших масштабах.

## 📝 Критический анализ

```markdown
# DPR-CTL (2020)
---
[[paper]](https://arxiv.org/abs/2004.04906)<br>
DPR = Dense Passage Retrieval (with Contrastive Training Logic)

__DPR__ — метод плотного информационного поиска, использующий Bi-Encoder архитектуру для сопоставления запросов и документов в семантическом векторном пространстве, решая проблему "лексического разрыва".

__Постановка задачи__<br>
В Open-Domain Question Answering (ODQA) требуется быстро извлечь релевантные пассажи из огромной коллекции документов.

__Мотивация__<br>
Традиционные системы, такие как BM25, не учитывают семантическое сходство. Требовалось решение, которое понимает семантику и работает быстро.

__Существующие подходы__<br>
- BM25: высокая скорость, но отсутствие семантики.
- Cross-Encoders: точные, но медленные.
- ORQA (2019): первая попытка Dense Retrieval, но сложная в обучении.

__Идея__<br>
Использовать Bi-Encoder архитектуру с Contrastive Learning, включая In-batch и Hard negatives, для различения семантически близких, но нерелевантных пассажей.

__Архитектура__<br>
- Question Encoder ($E_Q$): преобразует вопрос в вектор.
- Passage Encoder ($E_P$): преобразует пассаж в вектор.
- Similarity Function: скалярное произведение векторов.

<img src="img/img.png" width=500>

__Алгоритм обучения__<br>
Используется Contrastive Loss:
1. In-batch negatives: негативы — позитивные пассажи других вопросов в батче.
2. Hard negatives: пассажи с высоким скором по BM25, но без ответа.
3. Оптимизация: минимизация расстояния до $p^+$ и максимизация до $p^-$.

__Алгоритм инференса__<br>
1. Offline: кодирование коллекции документов через $E_P$ и индексация в FAISS.
2. Online: кодирование вопроса через $E_Q$.
3. Retrieval: Maximum Inner Product Search в FAISS-индексе.

__Результаты__<br>
На Natural Questions:
- Top-20 Retrieval Accuracy: 79.4% у DPR против 59.1% у BM25, прирост более 20пп.
- В связке с моделью-ридером точность ответов выросла на 9-11пп.
```


## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример иллюстрации метода Dense Passage Retrieval (DPR) с использованием библиотеки Hugging Face Transformers.
# Мы будем использовать предварительно обученные модели для кодирования вопросов и пассажей.

from transformers import DPRQuestionEncoder, DPRQuestionEncoderTokenizer
from transformers import DPRContextEncoder, DPRContextEncoderTokenizer
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Инициализация токенайзеров и моделей для вопроса и пассажа
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

passage_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
passage_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")

# Пример вопроса и пассажей
question = "Who developed the theory of relativity?"
passages = [
    "Albert Einstein developed the theory of relativity.",
    "Isaac Newton formulated the laws of motion.",
    "Marie Curie was a pioneer in the field of radioactivity."
]

# Кодирование вопроса
question_inputs = question_tokenizer(question, return_tensors="pt")
question_embedding = question_encoder(**question_inputs).pooler_output

# Кодирование пассажей
passage_embeddings = []
for passage in passages:
    passage_inputs = passage_tokenizer(passage, return_tensors="pt")
    passage_embedding = passage_encoder(**passage_inputs).pooler_output
    passage_embeddings.append(passage_embedding)

# Преобразование списка тензоров в один тензор
passage_embeddings = torch.cat(passage_embeddings, dim=0)

# Вычисление сходства между вопросом и каждым пассажем
similarities = cosine_similarity(question_embedding.detach().numpy(), passage_embeddings.detach().numpy())

# Вывод результатов
for i, passage in enumerate(passages):
    print(f"Passage: {passage}")
    print(f"Similarity: {similarities[0][i]:.4f}\n")

# В этом примере мы видим, как DPR позволяет находить семантически релевантные пассажи,
# даже если в них нет точного совпадения с запросом.
# В отличие от BM25, DPR использует плотные векторные представления, что позволяет
# учитывать семантическое сходство, а не только совпадение токенов.
```

Этот пример демонстрирует использование модели DPR для кодирования вопросов и пассажей в плотные векторные представления. Мы используем предварительно обученные модели из библиотеки Hugging Face Transformers. Вопрос и пассажи кодируются независимо, и затем мы вычисляем косинусное сходство между вектором вопроса и векторами пассажей. Это иллюстрирует ключевую концепцию DPR — использование Bi-Encoder архитектуры для семантического поиска.